# Apple (AAPL) Stock Price Prediction using LSTM

**Rhombix Technologies — Data Science Internship — Task 2: Stock Prediction**

This notebook downloads historical Apple Inc. (AAPL) stock price data, preprocesses it, and trains a Long Short-Term Memory (LSTM) neural network to predict future closing prices.

**Steps covered:**
1. Install & import libraries
2. Download AAPL historical data
3. Exploratory Data Analysis (EDA)
4. Data preprocessing (scaling, sequence creation)
5. Train/test split
6. Build & train LSTM model
7. Evaluate & visualize predictions
8. Predict the next day's price

## 1. Install & Import Libraries

Run the cell below once if you don't already have these packages installed.

In [ ]:
# Run this once to install required packages (uncomment if needed)
# !pip install yfinance tensorflow scikit-learn matplotlib pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
print('Libraries imported successfully')

## 2. Download AAPL Historical Data

We use `yfinance` to pull several years of daily historical prices for Apple (ticker: **AAPL**).

In [ ]:
TICKER = 'AAPL'
START_DATE = '2015-01-01'
END_DATE = None  # None = up to today

df = yf.download(TICKER, start=START_DATE, end=END_DATE)
df = df[['Close']].rename(columns={'Close': 'Close_Price'})
df.dropna(inplace=True)

print(df.shape)
df.tail()

## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['Close_Price'], color='royalblue', linewidth=1.2)
plt.title(f'{TICKER} Closing Price History')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.show()

In [ ]:
df.describe()

## 4. Data Preprocessing

We scale prices to the range [0, 1] (LSTMs train much better on normalized data) and build sequences: each input is the previous `WINDOW_SIZE` days of prices, and the target is the next day's price.

In [ ]:
WINDOW_SIZE = 60  # number of past days used to predict the next day

data = df['Close_Price'].values.reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

def create_sequences(dataset, window_size):
    X, y = [], []
    for i in range(window_size, len(dataset)):
        X.append(dataset[i - window_size:i, 0])
        y.append(dataset[i, 0])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, WINDOW_SIZE)
print('X shape:', X.shape, '| y shape:', y.shape)

## 5. Train/Test Split

We keep the data in chronological order (no shuffling) since this is time-series data — the last 20% is held out as the test set.

In [ ]:
split_ratio = 0.8
split_index = int(len(X) * split_ratio)

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

# LSTM expects input shape: (samples, timesteps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

## 6. Build & Train the LSTM Model

In [ ]:
model = Sequential([
    LSTM(units=64, return_sequences=True, input_shape=(X_train.shape[1], 1)),
    Dropout(0.2),
    LSTM(units=64, return_sequences=False),
    Dropout(0.2),
    Dense(units=32, activation='relu'),
    Dense(units=1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.show()

## 7. Evaluate & Visualize Predictions

In [ ]:
predicted_scaled = model.predict(X_test)

# Inverse-transform back to real price scale
predicted_prices = scaler.inverse_transform(predicted_scaled)
actual_prices = scaler.inverse_transform(y_test.reshape(-1, 1))

rmse = np.sqrt(mean_squared_error(actual_prices, predicted_prices))
mae = mean_absolute_error(actual_prices, predicted_prices)
mape = np.mean(np.abs((actual_prices - predicted_prices) / actual_prices)) * 100

print(f'RMSE: {rmse:.2f}')
print(f'MAE:  {mae:.2f}')
print(f'MAPE: {mape:.2f}%')

In [ ]:
test_dates = df.index[-len(actual_prices):]

plt.figure(figsize=(14, 6))
plt.plot(test_dates, actual_prices, label='Actual Price', color='royalblue')
plt.plot(test_dates, predicted_prices, label='Predicted Price', color='orangered')
plt.title(f'{TICKER} Stock Price: Actual vs Predicted (LSTM)')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.show()

## 8. Predict the Next Day's Closing Price

In [ ]:
last_window = scaled_data[-WINDOW_SIZE:]
last_window = last_window.reshape(1, WINDOW_SIZE, 1)

next_day_scaled = model.predict(last_window)
next_day_price = scaler.inverse_transform(next_day_scaled)

print(f'Predicted next closing price for {TICKER}: ${next_day_price[0][0]:.2f}')

## Conclusion

- We built an LSTM model that learns patterns from the past 60 days of AAPL closing prices to predict the next day's price.
- Model performance is reported via RMSE, MAE, and MAPE.
- The Actual vs Predicted chart shows how closely the model tracks real price movement.

**Note:** Stock prices are influenced by countless real-world factors (news, earnings, macroeconomics) that a price-only LSTM cannot capture. This model is for learning/demonstration purposes, not real trading advice.

**For the internship submission:**
1. Run this notebook fully end-to-end in Jupyter.
2. Push this `.ipynb` file to your GitHub repo `RhombixTechnologies_Tasks`.
3. Record a short video explaining the code and results, post it on LinkedIn tagging @Rhombix Technologies, and share the GitHub link.